# 22. Reproducible Analysis: Scripts, Pipelines and Tests

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `variants.py`

Annotate confident variants with the gene and amino acid they change.


In [ ]:
"""Annotate confident variants with the gene and amino acid they change.

Chapter 16's effects.py rebuilt as a command line tool: its inputs and
thresholds are arguments, its progress is logged, its result is a file,
and its logic is in functions that tests can call.

    uv run variants.py calls.vcf.gz genes.gff3 genome.fasta -o out.tsv
"""

import argparse
import csv
import logging

import pysam
from Bio import SeqIO
from Bio.Seq import Seq

log = logging.getLogger("variants")


def read_coding_sequences(gff_path: str) -> list[dict]:
    """CDS features from a GFF3 file, one based closed coordinates."""
    features = []
    with open(gff_path) as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            if fields[2] != "CDS":
                continue
            attributes = dict(
                pair.split("=", 1) for pair in fields[8].split(";")
            )
            features.append(
                {
                    "start": int(fields[3]),
                    "end": int(fields[4]),
                    "strand": fields[6],
                    "name": attributes.get("gene")
                    or attributes.get("product", "?")[:24],
                    "frameshift": "frameshift" in fields[8],
                }
            )
    return features


def amino_acid_change(
    gene: Seq, strand: str, start: int, end: int, position: int, alt: str
) -> str:
    """The change a one letter variant makes, such as D127N missense.

    gene is the sequence from start to end on the plus strand; start, end
    and position are one based, as in GFF and VCF.
    """
    offset = position - start
    if strand == "-":
        gene = gene.reverse_complement()
        offset = end - position
        alt = str(Seq(alt).reverse_complement())
    mutated = gene[:offset] + alt + gene[offset + 1 :]
    codon = offset // 3
    before = gene[3 * codon : 3 * codon + 3].translate(table=11)
    after = mutated[3 * codon : 3 * codon + 3].translate(table=11)
    if before == after:
        kind = "synonymous"
    elif after == "*":
        kind = "nonsense"
    else:
        kind = "missense"
    return f"{before}{codon + 1}{after} {kind}"


def annotate(vcf_path, gff_path, fasta_path, min_qual, min_depth):
    """Yield one row per variant that passes the thresholds."""
    genome = SeqIO.read(fasta_path, "fasta").seq
    coding = read_coding_sequences(gff_path)
    log.info("read %d coding sequences", len(coding))
    for record in pysam.VariantFile(vcf_path):
        if record.qual < min_qual or record.info["DP"] < min_depth:
            continue
        alt = record.alts[0]
        hits = [c for c in coding if c["start"] <= record.pos <= c["end"]]
        effect = "between genes"
        if hits:
            cds = hits[0]
            if cds["frameshift"]:
                effect = "frameshifted gene, not translated"
            elif len(record.ref) == 1 and len(alt) == 1:
                gene = genome[cds["start"] - 1 : cds["end"]]
                effect = amino_acid_change(
                    gene,
                    cds["strand"],
                    cds["start"],
                    cds["end"],
                    record.pos,
                    alt,
                )
            else:
                effect = "insertion or deletion"
            effect = f"{cds['name']}: {effect}"
        yield {
            "position": record.pos,
            "ref": record.ref,
            "alt": alt,
            "quality": round(record.qual),
            "effect": effect,
        }


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("vcf", help="variant calls, VCF or BCF")
    parser.add_argument("gff", help="gene annotation, GFF3")
    parser.add_argument("fasta", help="reference genome, FASTA")
    parser.add_argument("-o", "--output", required=True, help="TSV file")
    parser.add_argument("--min-qual", type=float, default=30)
    parser.add_argument("--min-depth", type=int, default=10)
    args = parser.parse_args()

    logging.basicConfig(
        level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s"
    )
    log.info(
        "thresholds: QUAL >= %s, depth >= %s",
        args.min_qual,
        args.min_depth,
    )
    rows = list(
        annotate(
            args.vcf, args.gff, args.fasta, args.min_qual, args.min_depth
        )
    )
    with open(args.output, "w", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=list(rows[0]), delimiter="\t"
        )
        writer.writeheader()
        writer.writerows(rows)
    log.info("wrote %d variants to %s", len(rows), args.output)


if __name__ == "__main__":
    main()

## `test_variants.py`

Tests for variants.py. Run with: uv run pytest


In [ ]:
"""Tests for variants.py. Run with: uv run pytest"""

from pathlib import Path

import pytest
from Bio.Seq import Seq

from variants import amino_acid_change, annotate

# A made-up gene of three codons, ATG GAT TAA (Met, Asp, stop), placed at
# positions 101 to 109 of a genome, counting from 1.
GENE = Seq("ATGGATTAA")


def test_missense_on_the_plus_strand():
    # position 104 is the G of GAT; GAT to AAT turns Asp into Asn
    assert (
        amino_acid_change(GENE, "+", 101, 109, 104, "A") == "D2N missense"
    )


def test_synonymous_change():
    # GAT to GAC still codes for Asp
    assert amino_acid_change(GENE, "+", 101, 109, 106, "C") == (
        "D2D synonymous"
    )


def test_nonsense_change():
    # ATG TGG TAA (Met, Trp, stop) at positions 1 to 9: changing the last
    # G of TGG, position 6, to A makes TGA, a stop codon
    gene = Seq("ATGTGGTAA")
    assert amino_acid_change(gene, "+", 1, 9, 6, "A") == "W2* nonsense"


def test_minus_strand_gene():
    # The same made-up gene written on the minus strand: the genome holds
    # its reverse complement, TTAATCCAT, at positions 101 to 109.
    genome_side = GENE.reverse_complement()
    # The gene's G of GAT (gene position 4) sits at genome position 106,
    # where the plus strand shows its complement, C. G>A on the gene is
    # C>T on the genome.
    assert amino_acid_change(genome_side, "-", 101, 109, 106, "T") == (
        "D2N missense"
    )


DATA = Path("../data/rel606.fasta")


@pytest.mark.skipif(not DATA.exists(), reason="needs data/fetch.py ch15")
def test_pykf_in_the_real_genome():
    rows = annotate(
        "../ch16/haploid.vcf.gz",
        "../ch15/rel606.gff3",
        str(DATA),
        min_qual=30,
        min_depth=10,
    )
    effects = {row["position"]: row["effect"] for row in rows}
    assert len(effects) == 25
    assert effects[1_733_343] == "pykF: D127N missense"

## `versions.py`

Record exactly which versions produced a result, for the methods section of a paper and for anyone who reruns the analysis.


In [ ]:
# Record exactly which versions produced a result, for the methods section
# of a paper and for anyone who reruns the analysis.
import platform
import subprocess
from importlib.metadata import version

PACKAGES = ["biopython", "pysam", "pandas", "numpy", "snakemake"]
TOOLS = {
    "minimap2": ["--version"],
    "samtools": ["--version"],
    "bcftools": ["--version"],
}

lines = [f"python {platform.python_version()}"]
lines += [f"{name} {version(name)}" for name in PACKAGES]
for tool, arguments in TOOLS.items():
    output = subprocess.run(
        [tool, *arguments],
        capture_output=True,
        text=True,
        errors="replace",  # one tool prints a byte that is not UTF-8
        check=True,
    ).stdout
    first = output.splitlines()[0].split()
    lines.append(f"{tool} {first[-1]}")

with open("versions.txt", "w") as handle:
    handle.write("\n".join(lines) + "\n")
print("\n".join(lines))